# STT – Version 2 : semi temps réel + arrêt vocal

Cette version améliore la V1 (STT basique) en ajoutant :

- une boucle **semi temps réel** avec des segments de 2 secondes,
- un filtrage simple des segments silencieux par énergie,
- un **arrêt vocal** naturel en disant le mot-clé **"stop"**.

La V1 ne faisait qu'un enregistrement unique suivi d'une transcription.
Ici, la transcription est continue : la parole est découpée en segments et transcrite progressivement.


In [ ]:
import os
import shutil
import numpy as np
import sounddevice as sd
import whisper

# Facultatif : s'assurer que ffmpeg est dans le PATH si tu veux aussi lire des fichiers
os.environ["PATH"] += r";C:\ffmpeg\ffmpeg-8.0-essentials_build\bin"
print("ffmpeg vu par Python ? ->", shutil.which("ffmpeg"))

# Paramètres audio
FS = 16000            # fréquence d'échantillonnage
SEGMENT_DURATION = 2  # durée d'un segment (s)
N_SAMPLES = FS * SEGMENT_DURATION


In [ ]:
model = whisper.load_model("small")
print("Modèle Whisper chargé.")


In [ ]:
def capture_and_transcribe_loop(language="fr", energy_threshold=0.01):
    """
    Enregistre et transcrit en continu, par segments de 2 secondes.
    Arrêt vocal : dire 'stop'.
    """
    print("Démarrage de la boucle semi temps réel (V2).")
    print("Parle par petites phrases (2–3 secondes).")
    print("Pour arrêter, dis simplement 'stop'.\n")
    
    segment_idx = 0

    try:
        while True:
            print(f"\n[Segment {segment_idx}] Enregistrement…")

            # Enregistrement d'un segment de 2 secondes
            audio = sd.rec(int(N_SAMPLES), samplerate=FS, channels=1, dtype="float32")
            sd.wait()

            # Mono
            audio_mono = audio[:, 0]

            # Énergie moyenne (pour éviter de transcrire le silence)
            energy = float(np.mean(np.abs(audio_mono)))
            print(f"Énergie moyenne du segment : {energy:.5f}")

            if energy < energy_threshold:
                print("[Segment ignoré : trop silencieux]")
                segment_idx += 1
                continue

            print("[Transcription en cours…]")
            result = model.transcribe(audio_mono, language=language, fp16=False)
            text = (result.get("text") or "").strip()

            if text:
                print(f"[Segment {segment_idx}] Texte reconnu : {text}")

                # ARRÊT VOCAL
                if "stop" in text.lower():
                    print("Commande vocale 'stop' détectée. Arrêt du système.")
                    break

                # 👉 plus tard : envoyer 'text' au module texte → signes
                # send_to_sign_module(text)

            else:
                print(f"[Segment {segment_idx}] Aucun texte reconnu.")

            segment_idx += 1

    except KeyboardInterrupt:
        print("\nArrêt manuel par l'utilisateur (Ctrl+C / bouton Stop).")

    print("Système V2 arrêté proprement.")


In [ ]:
capture_and_transcribe_loop(language="fr", energy_threshold=0.01)


## Améliorations possibles pour une V3

Idées d'amélioration pour une version future :

- utiliser un modèle Whisper plus léger (`base` ou `tiny`) pour réduire la latence,
- gérer plusieurs langues (détection automatique ou paramètre utilisateur),
- brancher directement la sortie texte sur le module « texte → gloss → signes »,
- ajouter une visualisation en temps réel (affichage des segments et du texte reconnu).

La V2 est la première version semi temps réel fonctionnelle, avec arrêt vocal intégré.
